In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from databricks.sdk import WorkspaceClient
from lh import common, publish

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
dbutils.widgets.dropdown("mode", "plan", ["plan", "apply"])
env, mode = dbutils.widgets.get("env"), dbutils.widgets.get("mode")
cfg = common.load_config(env)
root = os.path.dirname(os.getcwd())
bi_cfg = publish.load_bi(root)
rows = publish.run(publish.Api(WorkspaceClient()), bi_cfg, root, env, apply=(mode == "apply"))
print(f"{cfg['code']} {env} mode={mode} -> {bi_cfg['folder']}/{env}")
for r in rows:
    print(f"{r['dashboard']:24} {r['action']:7} viewers: {r['viewers']}")
if mode == "apply":
    common.log_run(spark, cfg, "bi_publish", "success", len(rows), str([r["dashboard"] for r in rows]))